### 0: Downloading Programmes

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
import pandas as pd
from pathlib import Path
from collections import defaultdict
import numpy as np


### 1: Dataframes

In [2]:
base = Path("output_folder")


df_resources = pd.read_csv(base / "resources.csv") # resource ID and resource type
df_constraints = pd.read_csv(base / "constraints.csv") # constraint ID and type, Name, required?, weight and cost function

df_times = pd.read_csv(base / "times.csv").sort_values("order_index") # timeslots and their timegroups (days)


# df_events = event with its duration, course, groups, role, classes and teacher)

events = pd.read_csv(base / "events.csv") 
event_groups = pd.read_csv(base / "event_eventgroup_membership.csv")
event_resources = pd.read_csv(base / "event_resources.csv")

# Events each group is part of
groups_per_event = (
    event_groups
    .groupby("event_id")["event_group_id"]
    .apply(list)
    .reset_index(name="groups")
)

# Teacher of each event
teachers_per_event = event_resources[event_resources["role"] == "Teacher"]
teachers_per_event = ( teachers_per_event[["event_id", "reference_resource_id"]]
    .groupby("event_id")["reference_resource_id"]
    .apply(list)
    .reset_index(name="teachers")
) 

#Classes in each event
classes_per_event = event_resources[event_resources["role"].astype(str).str.startswith("Class")]
classes_per_event = ( classes_per_event[["event_id", "reference_resource_id"]]
    .groupby("event_id")["reference_resource_id"]
    .apply(list)
    .reset_index(name="classes")
)



# Room of each event
room_per_event = event_resources[event_resources["role"] == "Room"]
room_per_event = (room_per_event[["event_id", "reference_resource_id"]]
    .groupby("event_id")["reference_resource_id"]
    .apply(list)
    .reset_index(name="room")
)

df_events = (
    events.rename(columns={"course_ref": "course_reference"})[["event_id", "duration", "course_reference"]]
    .merge(groups_per_event, on="event_id", how="left")
    #.merge(role_per_event, on="event_id", how="left")
    .merge(room_per_event, on="event_id", how="left")
    .merge(classes_per_event, on="event_id", how="left")
    .merge(teachers_per_event, on="event_id", how="left")
) # event with its duration, course, groups, role, classes and teacher)



resource_groups = pd.read_csv(base / "resource_group_membership.csv")

# Keep only what we need from memberships and aggregate group memberships
groups_per_resource = (
    resource_groups
    .groupby("resource_id")["resource_group_id"]
    .apply(list)
    .reset_index(name="resource_groups")
)

# Merge onto resources
df_res_with_groups = (
    df_resources[["resource_id", "resource_type_ref"]]
    .merge(groups_per_resource, on="resource_id", how="left")
)

# Replace NaN with empty list (resources that have no group membership)
df_res_with_groups["resource_groups"] = df_res_with_groups["resource_groups"].apply(
    lambda x: x if isinstance(x, list) else []
)

# Split into separate tables
df_teachers = df_res_with_groups[df_res_with_groups["resource_type_ref"] == "Teacher"].reset_index(drop=True) # teacher id, resource type and list of groups
df_classes  = df_res_with_groups[df_res_with_groups["resource_type_ref"] == "Class"].reset_index(drop=True) # class id, resource type and list of groups
df_rooms    = df_res_with_groups[df_res_with_groups["resource_type_ref"] == "Room"].reset_index(drop=True) # room id, resource type and list of groups


df_res_long = (
    df_resources[["resource_id", "resource_type_ref"]]
    .merge(resource_groups[["resource_id", "resource_group_id"]], on="resource_id", how="left")
    .rename(columns={"resource_group_id": "resource_group"})
)
df_teachers_long = df_res_long[df_res_long["resource_type_ref"] == "Teacher"] # teacher id, resource type and seperate groups
df_classes_long  = df_res_long[df_res_long["resource_type_ref"] == "Class"] # class id, resource type and seperate groups
df_rooms_long    = df_res_long[df_res_long["resource_type_ref"] == "Room"] # room id, resource type and seperate groups


df_constraint_applied = pd.read_csv(base / "constraint_applies_to.csv")
df_constraint_params = pd.read_csv(base / "constraint_params.csv")

df_constraint_params = df_constraint_params.merge(
    df_constraints[["constraint_id", "constraint_type"]],
    on="constraint_id",
    how="left"
)

df_constraint_params = df_constraint_params.merge(
    df_constraint_applied[["constraint_id", "reference"]],
    on="constraint_id",
    how="left"
)

In [3]:
#print(df_resources)
#print(df_constraints)

#print(df_times)
#print(df_events)
#print(df_rooms)
#print(df_teachers)
#print(df_classes)

#df_constraint_params

### 2: Model 1 setup and helpers

In [48]:
def as_list(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    return [x]

In [49]:
### Sets

E = df_events["event_id"].tolist() # All events
T = df_times["time_id"].tolist() # All timeslots
R = df_rooms["resource_id"].tolist() # All rooms

all_teachers = df_teachers["resource_id"].tolist()
all_classes = df_classes["resource_id"].tolist()


# event -> duration/teachers/classes (lists)
event_to_duration = dict(zip(df_events["event_id"], df_events["duration"].astype(int))) # duration of each event
event_to_teachers = dict(zip(df_events["event_id"], df_events["teachers"])) # teachers for each event
event_to_classes  = dict(zip(df_events["event_id"], df_events["classes"])) # classes for each event
event_to_room = dict(zip(df_events["event_id"], df_events["room"])) # Room for each event

# event -> room group of each event
#event_to_roomgroup = dict(zip(df_events["event_id"], df_events["role"])) 

# room -> room group of each room
room_groups = {"gr_Rooms_X", "gr_Rooms_Y", "gr_Rooms_Z"}
room_to_group = (
    df_rooms_long[df_rooms_long["resource_group"].isin(room_groups)]
    .set_index("resource_id")["resource_group"]
    .to_dict()
)

# room group -> rooms in that group
group_to_rooms = defaultdict(list)
for room, grp in room_to_group.items():   # room_to_group: room -> gr_Rooms_X/Y/Z
    group_to_rooms[grp].append(room)


T_index = {t:i for i,t in enumerate(T)}

# Keep only true unavailability constraints (exclude "NotPreferred")
df_unavail_times = df_constraint_params[
    (df_constraint_params["constraint_type"] == "AvoidUnavailableTimesConstraint") &
    (df_constraint_params["path"] == "Times/Time") &
    (~df_constraint_params["constraint_id"].astype(str).str.contains("NotPreferred", case=False, na=False))
].copy()[["reference", "value"]]

# resource -> set(unavailable times)
resource_unavailability = (
    df_unavail_times
    .groupby("reference")["value"]
    .agg(lambda s: set(s))
    .to_dict()
)


In [50]:
### Feasible Start times for events:

#T_index = {t:i for i,t in enumerate(T)}

time_to_day = dict(zip(df_times["time_id"], df_times["day_ref"]))

# Build feasible starts per event (no crossing day boundary) (implicitely covers constraints 5,9,10,11,12)
feasible_starts = {}
for e in E:
    d = int(event_to_duration[e]) # number of periods needed
    starts = []
    for ts in T: # for all possible start times
        i0 = T_index[ts]
        # must have enough periods remaining in the global list
        if i0 + d - 1 >= len(T):
            continue
        day0 = time_to_day[ts]
        block = T[i0:i0 + d]
        # enforce all times in the block are same day
        if all(time_to_day[t] == day0 for t in block):
            starts.append(ts)
    feasible_starts[e] = starts

# If event e starts at ts, then occ_times[(e, ts)] = [t1, t2, ...] is the list of all times occupied by that event
occ_times = {}
for e in E:
    d = int(event_to_duration[e])
    for ts in feasible_starts[e]:
        i0 = T_index[ts]
        occ_times[(e, ts)] = T[i0:i0 + d]


# Helper: all (e, ts) pairs that occupy a given time t
# ie for Fr_3, if event 1 is duration 1 and event 2 is duration 2, then (1, Fr_3) and (2, Fr_2) and (2, Fr_3) occupy it
occupies_at_time = {t: [] for t in T}
for e in E:
    for ts in feasible_starts[e]:
        for t in occ_times[(e, ts)]:
            occupies_at_time[t].append((e, ts))



In [51]:

# Filter feasible starts against both teacher and class unavailability
feasible_starts_filtered = {}

for e in E:
    starts_ok = []
    teachers = set(as_list(event_to_teachers[e]))
    classes  = set(as_list(event_to_classes[e]))
    resources_for_event = teachers | classes

    for ts in feasible_starts[e]:
        occ = occ_times[(e, ts)]
        violates = False

        for res in resources_for_event:
            unavail = resource_unavailability.get(res, set())
            if any(t in unavail for t in occ):
                violates = True
                break

        if not violates:
            starts_ok.append(ts)

    feasible_starts_filtered[e] = starts_ok

# Replace
feasible_starts = feasible_starts_filtered
#feasible_starts


# rebuild occ_times
occ_times = {}
for e in E:
    d = int(event_to_duration[e])
    for ts in feasible_starts[e]:
        i0 = T_index[ts]
        occ_times[(e, ts)] = T[i0:i0 + d]

# rebuild occupies_at_time
occupies_at_time = {t: [] for t in T}
for e in E:
    for ts in feasible_starts[e]:
        for t in occ_times[(e, ts)]:
            occupies_at_time[t].append((e, ts))

#feasible_starts

In [16]:
#### Hard Coding Feasible Starts

time_to_day = dict(zip(df_times["time_id"], df_times["day_ref"]))

all_starts = {e: list(T) for e in E}

# Build occupied-time blocks for every candidate start
occ_times = {}
for e in E:
    d = int(event_to_duration[e])
    for ts in T:
        i0 = T_index[ts]
        if i0 + d - 1 < len(T):
            occ_times[(e, ts)] = T[i0:i0 + d]
        else:
            occ_times[(e, ts)] = []


occupies_at_time = {t: [] for t in T}
for e in E:
    for ts in T:
        for t in occ_times[(e, ts)]:
            occupies_at_time[t].append((e, ts))

feasible_starts = all_starts



In [52]:
event_to_resources = (
    event_resources.groupby("event_id")["reference_resource_id"]
    .apply(list)
    .to_dict()
)


### 2b: Pyomo Model part 1: Sets and Decision Variables

Constraints:

Part 1: Event -> time: (hard constraints only)

1) AssignTimes: [gr_EventsGeneral] Every event must be assigned a time
2) Do Not Split Events: [gr_EventsGeneral] Events must inhabit consecutive periods
3) PreferredTimes: [gr_EventsDuration1] Events with duration 1 must only be allocated to times in groups gr_TimesDurationOne
3) PreferredTimesDurationTwo: [gr_EventsDuration1] Events with duration 2 must only be allocated to times in groups gr_TimesDurationTwo
4) EventSpreadingConstraint: [Selected groups] events from each goup have a maximum and minimum specified number of events in each time group [monday, tuesday etcc...] (here max 1 per day) (event group given in course ID)
5) NoResourceClashes: No clashes for any of the resources
6) Unavailabilities_teachers/classes: [selected teacher/class] selected Teacher cannot be allocated event at any of the given times,


Part 2: Event -> time: (with soft constraints)
1) JumpPeriods_Classes: [gr_classes] classes should have a minimum of 0 and maximum of 0 idle timeperiods (no timeperiod where there is no event but there is an earlier and later event on the same day) on every given time group (day) (soft: weight = 1, quadratic)
2) JumpPeriods_Teachers: [gr_teachers] teachers should have a minimum of 0 and maximum of 0 idle timeperiods (no timeperiod where there is no event but there is an earlier and later event on the same day) on every given time group (day) (soft: weight = 1, quadratic)
3) MinMaxADay_Classes: [Individual Classes] Classes should attend between min and max periods for each day (soft: weight = 1, quadratic)
4) MinMaxADay_[select_teachers]: [Individual Teachers] Teachers should teach between min and max periods for each day (soft: weight = 1, quadratic)
5) NotPrefferedTimes[teachers]: [individual_teacher] No teacher wants to teach on fr_p7

### 2b: Start-Time Based Model

In [53]:
# ---- Model ----
m1 = pyo.ConcreteModel()

# ---- Sets ----
m1.E = pyo.Set(initialize=E) # Set of events
m1.T = pyo.Set(initialize=T, ordered=True) # Set of timeslots (ordered)
m1.R = pyo.Set(initialize=R) # Set of rooms

m1.Teachers = pyo.Set(initialize=all_teachers) # set of all teachers
m1.Classes = pyo.Set(initialize=all_classes) # set of all classes

# standard set up
m1.X_index = pyo.Set(
    dimen=2,
    initialize=[(e, ts) for e in E for ts in feasible_starts[e]]
)



# ---- Decision Variables ----
# Start-time decision variables:
# x[e, ts, r] = 1 if event e starts at time ts in room r
# Only define variables for feasible starts (and all rooms) (THIS SPEEDS UP MODEL)
m1.x = pyo.Var(m1.X_index, domain=pyo.Binary)

In [54]:
# Standard set up

# 1) Each event scheduled exactly once (one start time)   (covers constraints 4)
def one_start_rule(m, e):
    return sum(m.x[e, ts] for ts in feasible_starts[e]) == 1
m1.OneStart = pyo.Constraint(m1.E, rule=one_start_rule)

# 2) No room conflicts: at most one event can occupy a room at a time
# No room clashes with fixed rooms
def room_conflict_rule(m, r, t):
    terms = [
        m.x[e, ts]
        for (e, ts) in occupies_at_time.get(t, [])
        if event_to_room.get(e) == r
    ]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1

m1.RoomConflict = pyo.Constraint(m1.R, m1.T, rule=room_conflict_rule)

# 3) No teacher conflicts: a teacher can teach at most one event at a time
def teacher_conflict_rule(m, teacher, t):
    terms = [
        m.x[e, ts]
        for (e, ts) in occupies_at_time.get(t, [])
        if teacher in as_list(event_to_teachers[e])
    ]
    if not terms:
        return pyo.Constraint.Skip   # or pyo.Constraint.Feasible
    return sum(terms) <= 1

m1.TeacherConflict = pyo.Constraint(all_teachers, m1.T, rule=teacher_conflict_rule)

# 4) No class conflicts: a class can attend at most one event at a time
def class_conflict_rule(m, cls, t):
    terms = [
        m.x[e, ts]
        for (e, ts) in occupies_at_time.get(t, [])
        if cls in event_to_classes[e]
    ]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1

m1.ClassConflict = pyo.Constraint(all_classes, m1.T, rule=class_conflict_rule)



In [55]:

### 3) Events that are part of the same course must be on different days (constraint 13)


# 1) Map event -> course_reference
event_to_course = dict(zip(df_events["event_id"], df_events["course_reference"]))

# 2) Build course groups (gr_C001 ... gr_C150)
courses = sorted(set(event_to_course[e] for e in E))

# 3) Days (time groups) from df_times (gr_Mo, gr_Tu, ...)
days = sorted(df_times["day_ref"].dropna().unique().tolist())

# 4) Collect relevant x vars by (course, day)
#    key -> list of all feasible (e, ts) combinations for given course and day
course_day_terms = defaultdict(list)

for (e, ts) in m1.X_index:
    c = event_to_course[e]
    d = time_to_day[ts]   # day of the start time
    course_day_terms[(c, d)].append((e, ts))

# 5) Define Pyomo sets for indexing
m1.COURSES = pyo.Set(initialize=courses)
m1.DAYS = pyo.Set(initialize=days)

# 6) Max-per-day constraint (Maximum = 1)
def event_spread_max_rule(m, c, d):
    terms = course_day_terms.get((c, d), [])
    if not terms:
        return pyo.Constraint.Skip
    return sum(m.x[e, ts] for (e, ts) in terms) <= 1 # at most one event for every course each day

m1.EventSpreadMax = pyo.Constraint(m1.COURSES, m1.DAYS, rule=event_spread_max_rule)



# Example from your data: min=0, max=1 for each day
#day_min = {d: 0 for d in days}
#day_max = {d: 1 for d in days}



#m.EventSpreadMin = pyo.Constraint(m.COURSES, m.DAYS, rule=event_spread_min_rule)
#m.EventSpreadMax = pyo.Constraint(m.COURSES, m.DAYS, rule=event_spread_max_rule)



In [ ]:
#### FOR EXPLICIT FEASIBLE STARTS

start_allowed = {}

for e in E:
    dur = int(event_to_duration[e])
    resources_e = set(as_list(event_to_resources[e]))

    for ts in T:
        i0 = T_index[ts]

        # enough periods left in horizon
        if i0 + dur - 1 >= len(T):
            start_allowed[(e, ts)] = 0
            continue

        block = T[i0:i0 + dur]
        day0 = time_to_day[ts]

        # must stay within one day
        if not all(time_to_day[t] == day0 for t in block):
            start_allowed[(e, ts)] = 0
            continue

        # resource unavailability
        violates_unavailability = False
        for r in resources_e:
            if r in resource_unavailability:
                if any(t in resource_unavailability[r] for t in block):
                    violates_unavailability = True
                    break

        start_allowed[(e, ts)] = 0 if violates_unavailability else 1


m1.start_allowed = pyo.Param(
    m1.X_index,
    initialize=start_allowed,
    within=pyo.Binary
)

def valid_start_rule(m, e, ts):
    return m.x[e, ts] <= m.start_allowed[e, ts]

m1.ValidStart = pyo.Constraint(m1.X_index, rule=valid_start_rule)


In [56]:
# ---- Objective ---- (hard constraints only)
m1.obj = pyo.Objective(expr=0, sense=pyo.minimize)

In [57]:
# Preprocessing
seeds = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

solve_times = []
objectives = []
num_vars = []
num_cons = []
num_nonzeros = []
solve_status = []
start_x_list = []

import time as time_module

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(m1)

    solver = pyo.SolverFactory("gurobi")
    solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 1

    # Manual timing
    start_time = time_module.time()
    res = solver.solve(m1, tee=False)
    elapsed_time = time_module.time() - start_time

    solve_times.append(elapsed_time)
    objectives.append(pyo.value(m1.obj))
    num_vars.append(m1.nvariables())
    num_cons.append(m1.nconstraints())

    try:
        num_nonzeros.append(res.problem.number_of_nonzeros)
    except:
        num_nonzeros.append(None)

    solve_status.append(res.solver.termination_condition)

    # Store solution
    seed_solution = {}
    for (e, ts) in m1.X_index:
        val = pyo.value(m1.x[e, ts], exception=False)
        seed_solution[(e, ts)] = 0.0 if val is None else float(val)
    start_x_list.append(seed_solution)

# Keep compatibility with later cells that expect start_x / res
start_x = start_x_list[0]
res = res

# Calculate averages
valid_nonzeros = [n for n in num_nonzeros if n is not None]
avg_time = np.mean(solve_times)
avg_obj = np.mean(objectives)
avg_vars = np.mean(num_vars)
avg_cons = np.mean(num_cons)
avg_nonzeros = np.mean(valid_nonzeros) if valid_nonzeros else "Not available"

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}")
print(f"Average Objective: {avg_obj:.6f}")
print(f"Average Variables: {avg_vars:.0f}")
print(f"Average Constraints: {avg_cons:.0f}")
print(f"Average Nonzeros: {avg_nonzeros if isinstance(avg_nonzeros, str) else f'{avg_nonzeros:.0f}'}")
print(f"Termination Status: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_x_list):
    scheduled = sum(1 for v in seed_sol.values() if v > 0)
    print(f"  Seed {seeds[i]}: {scheduled} events scheduled in {solve_times[i]:.2f}s, Obj={objectives[i]:.6f}")


SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 0.2662
Average Objective: 0.000000
Average Variables: 5735
Average Constraints: 1927
Average Nonzeros: 23650
Termination Status: {<TerminationCondition.optimal: 'optimal'>}

Phase 1 produced 10 solutions (one per seed)
  Seed 10: 280 events scheduled in 0.33s, Obj=0.000000
  Seed 20: 280 events scheduled in 0.23s, Obj=0.000000
  Seed 30: 280 events scheduled in 0.62s, Obj=0.000000
  Seed 40: 280 events scheduled in 0.20s, Obj=0.000000
  Seed 50: 280 events scheduled in 0.20s, Obj=0.000000
  Seed 60: 280 events scheduled in 0.21s, Obj=0.000000
  Seed 70: 280 events scheduled in 0.22s, Obj=0.000000
  Seed 80: 280 events scheduled in 0.22s, Obj=0.000000
  Seed 90: 280 events scheduled in 0.21s, Obj=0.000000
  Seed 100: 280 events scheduled in 0.22s, Obj=0.000000


In [ ]:
# Explicit 
seeds = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

solve_times = []
objectives = []
num_vars = []
num_cons = []
num_nonzeros = []
solve_status = []
start_x_list = []

import time as time_module

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(m1)

    solver = pyo.SolverFactory("gurobi")
    solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 1

    # Manual timing
    start_time = time_module.time()
    res = solver.solve(m1, tee=False)
    elapsed_time = time_module.time() - start_time

    solve_times.append(elapsed_time)
    objectives.append(pyo.value(m1.obj))
    num_vars.append(m1.nvariables())
    num_cons.append(m1.nconstraints())

    try:
        num_nonzeros.append(res.problem.number_of_nonzeros)
    except:
        num_nonzeros.append(None)

    solve_status.append(res.solver.termination_condition)

    # Store solution
    seed_solution = {}
    for (e, ts) in m1.X_index:
        val = pyo.value(m1.x[e, ts], exception=False)
        seed_solution[(e, ts)] = 0.0 if val is None else float(val)
    start_x_list.append(seed_solution)

# Keep compatibility with later cells that expect start_x / res
start_x = start_x_list[0]
res = res

# Calculate averages
valid_nonzeros = [n for n in num_nonzeros if n is not None]
avg_time = np.mean(solve_times)
avg_obj = np.mean(objectives)
avg_vars = np.mean(num_vars)
avg_cons = np.mean(num_cons)
avg_nonzeros = np.mean(valid_nonzeros) if valid_nonzeros else "Not available"

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}")
print(f"Average Objective: {avg_obj:.6f}")
print(f"Average Variables: {avg_vars:.0f}")
print(f"Average Constraints: {avg_cons:.0f}")
print(f"Average Nonzeros: {avg_nonzeros if isinstance(avg_nonzeros, str) else f'{avg_nonzeros:.0f}'}")
print(f"Termination Status: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_x_list):
    scheduled = sum(1 for v in seed_sol.values() if v > 0)
    print(f"  Seed {seeds[i]}: {scheduled} events scheduled in {solve_times[i]:.2f}s, Obj={objectives[i]:.6f}")


SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 1.2358
Average Objective: 0.000000
Average Variables: 9800
Average Constraints: 12160
Average Nonzeros: 50716
Termination Status: {<TerminationCondition.optimal: 'optimal'>}

Phase 1 produced 10 solutions (one per seed)
  Seed 10: 280 events scheduled in 1.04s, Obj=0.000000
  Seed 20: 280 events scheduled in 0.85s, Obj=0.000000
  Seed 30: 280 events scheduled in 0.93s, Obj=0.000000
  Seed 40: 280 events scheduled in 1.06s, Obj=0.000000
  Seed 50: 280 events scheduled in 0.94s, Obj=0.000000
  Seed 60: 280 events scheduled in 1.10s, Obj=0.000000
  Seed 70: 280 events scheduled in 0.88s, Obj=0.000000
  Seed 80: 280 events scheduled in 1.50s, Obj=0.000000
  Seed 90: 280 events scheduled in 2.07s, Obj=0.000000
  Seed 100: 280 events scheduled in 1.98s, Obj=0.000000


### Hybrid

In [ ]:
# Build
'''
covering_starts = {(e, t): [] for e in E for t in T}
occ_times = {}

for e in E:
    dur = int(event_to_duration[e])
    for ts in T:
        i0 = T_index[ts]
        if i0 + dur - 1 < len(T):
            block = T[i0:i0 + dur]
            occ_times[(e, ts)] = block
            for t in block:
                covering_starts[(e, t)].append(ts)
        else:
            occ_times[(e, ts)] = []

'''
covering_starts = {(e, t): [] for e in E for t in T}  # preprocessing version
occ_times = {}

for e in E:
    dur = int(event_to_duration[e])
    for ts in feasible_starts[e]:
        i0 = T_index[ts]
        block = T[i0:i0 + dur]
        occ_times[(e, ts)] = block
        for t in block:
            covering_starts[(e, t)].append(ts)



# Hybrid
mb = pyo.ConcreteModel()

mb.E = pyo.Set(initialize=E)
mb.T = pyo.Set(initialize=T, ordered=True)

# all candidate starts
mb.Y_index = pyo.Set(
    dimen=2,
    initialize=[(e, ts) for e in E for ts in feasible_starts[e]]
)

mb.y = pyo.Var(mb.Y_index, domain=pyo.Binary)

# occupancy indicator
mb.w = pyo.Var(mb.E, mb.T, domain=pyo.Binary)



'''
# Explicit start legality 
# - enough periods remain
# - event must stay within one day
# - no assigned resource may be unavailable
start_allowed = {}

for e in E:
    dur = int(event_to_duration[e])
    resources_e = set(as_list(event_to_resources[e]))

    for ts in T:
        i0 = T_index[ts]

        if i0 + dur - 1 >= len(T):
            start_allowed[(e, ts)] = 0
            continue

        block = T[i0:i0 + dur]
        day0 = time_to_day[ts]

        if not all(time_to_day[t] == day0 for t in block):
            start_allowed[(e, ts)] = 0
            continue

        violates_unavailability = False
        for r in resources_e:
            if r in resource_unavailability:
                if any(t in resource_unavailability[r] for t in block):
                    violates_unavailability = True
                    break

        start_allowed[(e, ts)] = 0 if violates_unavailability else 1

mb.start_allowed = pyo.Param(
    mb.Y_index,
    initialize=start_allowed,
    within=pyo.Binary
)

def valid_start_rule(mb, e, ts):
    return mb.y[e, ts] <= mb.start_allowed[e, ts]
mb.ValidStart = pyo.Constraint(mb.Y_index, rule=valid_start_rule)
'''

# Each
def one_start_rule(mb, e):
    return sum(mb.y[e, ts] for ts in feasible_starts[e]) == 1
mb.OneStart = pyo.Constraint(mb.E, rule=one_start_rule)


# Link
def occupancy_link_rule(mb, e, t):
    starts_that_cover_t = covering_starts[(e, t)]
    if not starts_that_cover_t:
        return mb.w[e, t] == 0
    return mb.w[e, t] == sum(mb.y[e, ts] for ts in starts_that_cover_t)
mb.OccupancyLink = pyo.Constraint(mb.E, mb.T, rule=occupancy_link_rule)


# No
def teacher_clash_rule(mb, teacher, t):
    terms = [mb.w[e, t] for e in mb.E if teacher in as_list(event_to_teachers[e])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
mb.TeacherClash = pyo.Constraint(all_teachers, mb.T, rule=teacher_clash_rule)


# No
def class_clash_rule(mb, cls, t):
    terms = [mb.w[e, t] for e in mb.E if cls in as_list(event_to_classes[e])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
mb.ClassClash = pyo.Constraint(all_classes, mb.T, rule=class_clash_rule)


# Room
#roomgroup_cap = {
#    "gr_Room_X": 1,
#    "gr_Room_Y": 2,
#    "gr_Room_Z": 9,
#}

#mb.G = pyo.Set(initialize=roomgroup_cap.keys())



# Events
event_to_course = dict(zip(df_events["event_id"], df_events["course_reference"]))
courses = sorted(set(event_to_course[e] for e in E))
days = sorted(df_times["day_ref"].dropna().unique().tolist())

course_day_terms = defaultdict(list)
for (e, ts) in mb.Y_index:
    c = event_to_course[e]
    d = time_to_day[ts]
    course_day_terms[(c, d)].append((e, ts))

mb.COURSES = pyo.Set(initialize=courses)
mb.DAYS = pyo.Set(initialize=days)

def event_spread_max_rule(mb, c, d):
    terms = course_day_terms.get((c, d), [])
    if not terms:
        return pyo.Constraint.Skip
    return sum(mb.y[e, ts] for (e, ts) in terms) <= 1
mb.EventSpreadMax = pyo.Constraint(mb.COURSES, mb.DAYS, rule=event_spread_max_rule)


# Objective
mb.obj = pyo.Objective(expr=0, sense=pyo.minimize)


In [59]:
### Preprocessing 
seeds = [10, 20, 30, 40, 50, 60 ,70 ,80, 90, 100]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []
start_x_list = []

import time as time_module

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(mb)

    solver = pyo.SolverFactory("gurobi")
    solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 1

    # Manual timing
    start_time = time_module.time()
    res = solver.solve(mb, tee=False)
    elapsed_time = time_module.time() - start_time

    solve_times.append(elapsed_time)
    objectives.append(pyo.value(mb.obj))
    num_vars.append(mb.nvariables())
    num_cons.append(mb.nconstraints())

    solve_status.append(res.solver.termination_condition)

    # Store solution
    seed_solution = {}
    for (e, ts) in mb.Y_index:
        val = pyo.value(mb.y[e, ts], exception=False)
        seed_solution[(e, ts)] = 0.0 if val is None else float(val)
    start_x_list.append(seed_solution)

# Keep compatibility with later cells that expect start_x / res
start_x = start_x_list[0]
res = res

# Calculate averages
avg_time = sum(solve_times) / len(solve_times)
avg_obj = sum(objectives) / len(objectives)
avg_vars = sum(num_vars) / len(num_vars)
avg_cons = sum(num_cons) / len(num_cons)

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}")
print(f"Average Objective: {avg_obj:.6f}")
print(f"Average Variables: {avg_vars:.0f}")
print(f"Average Constraints: {avg_cons:.0f}")
print(f"Termination Status: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_x_list):
    scheduled = sum(1 for v in seed_sol.values() if v > 0)
    print(f"  Seed {seeds[i]}: {scheduled} events scheduled in {solve_times[i]:.2f}s, Obj={objectives[i]:.6f}")


SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 1.4084
Average Objective: 0.000000
Average Variables: 15535
Average Constraints: 12102
Termination Status: {<TerminationCondition.optimal: 'optimal'>}

Phase 1 produced 10 solutions (one per seed)
  Seed 10: 280 events scheduled in 2.29s, Obj=0.000000
  Seed 20: 280 events scheduled in 1.75s, Obj=0.000000
  Seed 30: 280 events scheduled in 1.61s, Obj=0.000000
  Seed 40: 280 events scheduled in 1.21s, Obj=0.000000
  Seed 50: 280 events scheduled in 0.99s, Obj=0.000000
  Seed 60: 280 events scheduled in 0.91s, Obj=0.000000
  Seed 70: 280 events scheduled in 1.21s, Obj=0.000000
  Seed 80: 280 events scheduled in 1.68s, Obj=0.000000
  Seed 90: 280 events scheduled in 1.05s, Obj=0.000000
  Seed 100: 280 events scheduled in 1.39s, Obj=0.000000


In [ ]:
### Explicit Constraints
seeds = [10, 20, 30, 40, 50, 60 ,70 ,80, 90, 100]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []
start_x_list = []

import time as time_module

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(m1)

    solver = pyo.SolverFactory("gurobi")
    solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 1

    # Manual timing
    start_time = time_module.time()
    res = solver.solve(mb, tee=False)
    elapsed_time = time_module.time() - start_time

    solve_times.append(elapsed_time)
    objectives.append(pyo.value(mb.obj))
    num_vars.append(mb.nvariables())
    num_cons.append(mb.nconstraints())

    solve_status.append(res.solver.termination_condition)

    # Store solution
    seed_solution = {}
    for (e, ts) in mb.Y_index:
        val = pyo.value(mb.y[e, ts], exception=False)
        seed_solution[(e, ts)] = 0.0 if val is None else float(val)
    start_x_list.append(seed_solution)

# Keep compatibility with later cells that expect start_x / res
start_x = start_x_list[0]
res = res

# Calculate averages
avg_time = sum(solve_times) / len(solve_times)
avg_obj = sum(objectives) / len(objectives)
avg_vars = sum(num_vars) / len(num_vars)
avg_cons = sum(num_cons) / len(num_cons)

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}")
print(f"Average Objective: {avg_obj:.6f}")
print(f"Average Variables: {avg_vars:.0f}")
print(f"Average Constraints: {avg_cons:.0f}")
print(f"Termination Status: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_x_list):
    scheduled = sum(1 for v in seed_sol.values() if v > 0)
    print(f"  Seed {seeds[i]}: {scheduled} events scheduled in {solve_times[i]:.2f}s, Obj={objectives[i]:.6f}")


SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 1.5937
Average Objective: 0.000000
Average Variables: 19600
Average Constraints: 21960
Termination Status: {<TerminationCondition.optimal: 'optimal'>}

Phase 1 produced 10 solutions (one per seed)
  Seed 10: 280 events scheduled in 1.28s, Obj=0.000000
  Seed 20: 280 events scheduled in 1.89s, Obj=0.000000
  Seed 30: 280 events scheduled in 2.18s, Obj=0.000000
  Seed 40: 280 events scheduled in 1.47s, Obj=0.000000
  Seed 50: 280 events scheduled in 1.66s, Obj=0.000000
  Seed 60: 280 events scheduled in 1.48s, Obj=0.000000
  Seed 70: 280 events scheduled in 1.63s, Obj=0.000000
  Seed 80: 280 events scheduled in 1.59s, Obj=0.000000
  Seed 90: 280 events scheduled in 1.57s, Obj=0.000000
  Seed 100: 280 events scheduled in 1.19s, Obj=0.000000


### Pure Occupancy

In [37]:
# Day
times_by_day = {}
for _, row in df_times.sort_values("order_index").iterrows():
    d = row["day_ref"]
    times_by_day.setdefault(d, []).append(row["time_id"])

days = list(times_by_day.keys())

prev_in_day = {}
next_in_day = {}

for d in days:
    day_times = times_by_day[d]
    for i, t in enumerate(day_times):
        prev_in_day[t] = None if i == 0 else day_times[i - 1]
        next_in_day[t] = None if i == len(day_times) - 1 else day_times[i + 1]


# -------------------------------------------------
# Build illegal starts explicitly
# - not enough room left in horizon
# - crosses day boundary
# - hits resource unavailable times
illegal_starts = []

for e in E:
    dur = int(event_to_duration[e])
    resources_e = set(as_list(event_to_resources[e]))

    for ts in T:
        i0 = T_index[ts]

        if i0 + dur - 1 >= len(T):
            continue

        block = T[i0:i0 + dur]
        day0 = time_to_day[ts]

        same_day = all(time_to_day[t] == day0 for t in block)

        violates_unavailability = False
        for r in resources_e:
            if r in resource_unavailability:
                if any(t in resource_unavailability[r] for t in block):
                    violates_unavailability = True
                    break

        if (not same_day) or violates_unavailability:
            illegal_starts.append((e, ts))


# Pure
mc = pyo.ConcreteModel()

mc.E = pyo.Set(initialize=E)
mc.T = pyo.Set(initialize=T, ordered=True)
mc.D = pyo.Set(initialize=days)

mc.w = pyo.Var(mc.E, mc.T, domain=pyo.Binary)

mc.U_index = pyo.Set(
    dimen=3,
    initialize=[(e, d, t) for e in E for d in days for t in times_by_day[d]]
)
mc.u = pyo.Var(mc.U_index, domain=pyo.Binary)
mc.v = pyo.Var(mc.U_index, domain=pyo.Binary)


# Exact
def duration_rule(mc, e):
    return sum(mc.w[e, t] for t in mc.T) == int(event_to_duration[e])
mc.Duration = pyo.Constraint(mc.E, rule=duration_rule)


# Explicit
mc.ILLEGAL_STARTS = pyo.Set(dimen=2, initialize=illegal_starts)

def illegal_block_rule(mc, e, ts):
    dur = int(event_to_duration[e])
    i0 = T_index[ts]
    block = T[i0:i0 + dur]
    return sum(mc.w[e, t] for t in block) <= dur - 1
mc.IllegalBlock = pyo.Constraint(mc.ILLEGAL_STARTS, rule=illegal_block_rule)


# No
def teacher_clash_rule(mc, teacher, t):
    terms = [mc.w[e, t] for e in mc.E if teacher in as_list(event_to_teachers[e])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
mc.TeacherClash = pyo.Constraint(all_teachers, mc.T, rule=teacher_clash_rule)


# No
def class_clash_rule(mc, cls, t):
    terms = [mc.w[e, t] for e in mc.E if cls in as_list(event_to_classes[e])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
mc.ClassClash = pyo.Constraint(all_classes, mc.T, rule=class_clash_rule)


# Room
#roomgroup_cap = {
#    "gr_Room_X": 1,
#    "gr_Room_Y": 2,
#    "gr_Room_Z": 9,
#}

#mc.G = pyo.Set(initialize=roomgroup_cap.keys())



# Contiguity
def start_edge_rule(mc, e, d, t):
    p = prev_in_day[t]
    if p is None:
        return mc.u[e, d, t] >= mc.w[e, t]
    return mc.u[e, d, t] >= mc.w[e, t] - mc.w[e, p]
mc.StartEdge = pyo.Constraint(mc.U_index, rule=start_edge_rule)

def end_edge_rule(mc, e, d, t):
    n = next_in_day[t]
    if n is None:
        return mc.v[e, d, t] >= mc.w[e, t]
    return mc.v[e, d, t] >= mc.w[e, t] - mc.w[e, n]
mc.EndEdge = pyo.Constraint(mc.U_index, rule=end_edge_rule)

def one_start_per_day_rule(mc, e, d):
    return sum(mc.u[e, d, t] for t in times_by_day[d]) <= 1
mc.OneStartPerDay = pyo.Constraint(mc.E, mc.D, rule=one_start_per_day_rule)

def one_end_per_day_rule(mc, e, d):
    return sum(mc.v[e, d, t] for t in times_by_day[d]) <= 1
mc.OneEndPerDay = pyo.Constraint(mc.E, mc.D, rule=one_end_per_day_rule)

def one_start_overall_rule(mc, e):
    return sum(mc.u[e, d, t] for d in mc.D for t in times_by_day[d]) == 1
mc.OneStartOverall = pyo.Constraint(mc.E, rule=one_start_overall_rule)

def one_end_overall_rule(mc, e):
    return sum(mc.v[e, d, t] for d in mc.D for t in times_by_day[d]) == 1
mc.OneEndOverall = pyo.Constraint(mc.E, rule=one_end_overall_rule)


# Events
mc.day_used = pyo.Var(mc.E, mc.D, domain=pyo.Binary)

def day_used_upper_rule(mc, e, d):
    return sum(mc.w[e, t] for t in times_by_day[d]) <= len(times_by_day[d]) * mc.day_used[e, d]
mc.DayUsedUpper = pyo.Constraint(mc.E, mc.D, rule=day_used_upper_rule)

def day_used_lower_rule(mc, e, d):
    return sum(mc.w[e, t] for t in times_by_day[d]) >= mc.day_used[e, d]
mc.DayUsedLower = pyo.Constraint(mc.E, mc.D, rule=day_used_lower_rule)

event_to_course = dict(zip(df_events["event_id"], df_events["course_reference"]))
courses = sorted(set(event_to_course[e] for e in E))

mc.COURSES = pyo.Set(initialize=courses)

def event_spread_max_rule(mc, c, d):
    terms = [mc.day_used[e, d] for e in mc.E if event_to_course[e] == c]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
mc.EventSpreadMax = pyo.Constraint(mc.COURSES, mc.D, rule=event_spread_max_rule)


# Objective
mc.obj = pyo.Objective(expr=0, sense=pyo.minimize)


In [ ]:
### Explicit Constraints 
seeds = [10, 20, 30, 40, 50, 60 ,70 ,80, 90, 100]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []
start_x_list = []

import time as time_module

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(mc)

    solver = pyo.SolverFactory("gurobi")
    solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 1

    # Manual timing
    start_time = time_module.time()
    res = solver.solve(mc, tee=False)
    elapsed_time = time_module.time() - start_time

    solve_times.append(elapsed_time)
    objectives.append(pyo.value(mc.obj))
    num_vars.append(mc.nvariables())
    num_cons.append(mc.nconstraints())

    solve_status.append(res.solver.termination_condition)


    # Store occupancy solution
    seed_solution = {}
    for e in mc.E:
        for t in mc.T:
            val = pyo.value(mc.w[e, t], exception=False)
            seed_solution[(e, t)] = 0.0 if val is None else float(val)
    start_x_list.append(seed_solution)

# Keep compatibility with later cells that expect start_x / res
start_x = start_x_list[0]
res = res

# Calculate averages
avg_time = sum(solve_times) / len(solve_times)
avg_obj = sum(objectives) / len(objectives)
avg_vars = sum(num_vars) / len(num_vars)
avg_cons = sum(num_cons) / len(num_cons)

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}")
print(f"Average Objective: {avg_obj:.6f}")
print(f"Average Variables: {avg_vars:.0f}")
print(f"Average Constraints: {avg_cons:.0f}")
print(f"Termination Status: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_x_list):
    scheduled = sum(1 for v in seed_sol.values() if v > 0)
    print(f"  Seed {seeds[i]}: {scheduled} events scheduled in {solve_times[i]:.2f}s, Obj={objectives[i]:.6f}")


SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 3.3591
Average Objective: 0.000000
Average Variables: 30800
Average Constraints: 32203
Termination Status: {<TerminationCondition.optimal: 'optimal'>}

Phase 1 produced 10 solutions (one per seed)
  Seed 10: 306 events scheduled in 5.47s, Obj=0.000000
  Seed 20: 306 events scheduled in 3.12s, Obj=0.000000
  Seed 30: 306 events scheduled in 4.49s, Obj=0.000000
  Seed 40: 306 events scheduled in 3.99s, Obj=0.000000
  Seed 50: 306 events scheduled in 2.81s, Obj=0.000000
  Seed 60: 306 events scheduled in 3.19s, Obj=0.000000
  Seed 70: 306 events scheduled in 2.89s, Obj=0.000000
  Seed 80: 306 events scheduled in 2.31s, Obj=0.000000
  Seed 90: 306 events scheduled in 2.69s, Obj=0.000000
  Seed 100: 306 events scheduled in 2.63s, Obj=0.000000


### Solve

In [36]:
# ---- Solve ---

solver = pyo.SolverFactory("gurobi")
solver.options["TimeLimit"] = 1000
# solver.options["MIPGap"] = 0.0

res = solver.solve(m1, tee=True)

print(res.solver.termination_condition) 

Read LP format model from file C:\Users\asher\AppData\Local\Temp\tmpaoph_6f1.pyomo.lp
Reading time = 0.05 seconds
x1: 1927 rows, 5736 columns, 23650 nonzeros
Set parameter TimeLimit to value 1000
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1000

Optimize a model with 1927 rows, 5736 columns and 23650 nonzeros (Min)
Model fingerprint: 0x549c9ad5
Model has 0 linear objective coefficients
Variable types: 1 continuous, 5735 integer (5735 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 0.0000000

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 8 available 

In [37]:
start_x = {}
for idx in m1.X_index:
    v = pyo.value(m1.x[idx], exception=False)
    start_x[idx] = 0.0 if v is None else float(v)

#start_x